# EBRAINS BrainScaleS-2 TTFS neuron-pooling experiment

Use the `EBRAINS-experimental` kernel and execute cells in order. This notebook is a thin launcher for the repository CLI. It deliberately does **not** install the full project `requirements.txt`, upgrade `torch`, or install `hxtorch`.

## 1. Install only notebook-side dependencies

`%pip` targets the active Jupyter kernel. The BrainScaleS-2 software stack remains the version supplied by EBRAINS.

In [ ]:
%pip install --quiet --disable-pip-version-check jaxtyping matplotlib

## 2. Locate the repository and select run stages

Hardware stages default to disabled so `Run All` cannot accidentally consume shared hardware time. Set the desired flags to `True`, then rerun this cell and the hardware-client setup cell.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

start = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (start, *start.parents)
        if (candidate / "scripts/evaluation/brainscales2_pooling.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the delayed-temporal repository root")
os.chdir(repo_root)

RUN_MOCK_SMOKE = True
RUN_HARDWARE_SMOKE = False
RUN_OPERATING_POINT_SWEEP = False
RUN_FULL_EXPERIMENT = False

# Formal full runs require an explicit, immutable .pbin calibration.
CALIBRATION_PATH = None  # Example: Path("/mnt/user/shared/calibration/spiking_calibration.pbin")
DEMOS_ROOT = repo_root.parent / "brainscales2-demos"
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ARTIFACT_ROOT = repo_root / "artifacts/brainscales2" / RUN_LABEL
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("repository:", repo_root)
print("python:", sys.executable)
print("artifacts:", ARTIFACT_ROOT)

## 3. Check the active EBRAINS kernel

In [ ]:
import inspect
import torch
import jaxtyping
import hxtorch
import hxtorch.spiking as hxsnn

print("Python:", sys.version)
print("torch:", torch.__version__)
print("hxtorch:", getattr(hxtorch, "__version__", "unknown"))
print("Experiment:", inspect.signature(hxsnn.Experiment))
print("Synapse:", inspect.signature(hxsnn.Synapse))
print("LIF:", inspect.signature(hxsnn.LIF))
print("run:", inspect.signature(hxsnn.run))

## 4. Configure the EBRAINS hardware client

When any hardware stage is enabled, this cell clones the official experimental demos beside the project if necessary, then calls the supported `setup_hardware_client()` helper. It does not print authentication environment variables.

In [ ]:
hardware_requested = any(
    (
        RUN_HARDWARE_SMOKE,
        RUN_OPERATING_POINT_SWEEP,
        RUN_FULL_EXPERIMENT,
    )
)

if hardware_requested:
    if not DEMOS_ROOT.is_dir():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "jupyter-notebooks-experimental",
                "https://github.com/electronicvisions/brainscales2-demos.git",
                str(DEMOS_ROOT),
            ],
            check=True,
        )
    if str(DEMOS_ROOT) not in sys.path:
        sys.path.insert(0, str(DEMOS_ROOT))
    from _static.common.helpers import setup_hardware_client
    setup_hardware_client()
    print("EBRAINS hardware client configured")
else:
    print("Hardware stages disabled; client setup skipped")

## 5. CLI helpers

Every CLI subprocess uses the active kernel's exact Python executable and inherits the hardware-client environment.

In [ ]:
CLI = repo_root / "scripts/evaluation/brainscales2_pooling.py"
VERIFY = repo_root / "scripts/verification/verify_brainscales2_pooling.py"

def run_python(script: Path, *arguments: object) -> None:
    command = [sys.executable, str(script), *(str(value) for value in arguments)]
    print(" ".join(command))
    subprocess.run(command, cwd=repo_root, check=True)

def show_artifacts(output_dir: Path) -> None:
    manifest_path = output_dir / "manifest.json"
    summary_path = output_dir / "summary.csv"
    if not manifest_path.is_file() or not summary_path.is_file():
        print("No completed artifacts:", output_dir)
        return
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    with summary_path.open(newline="", encoding="utf-8") as handle:
        summary = list(csv.DictReader(handle))
    print(json.dumps(
        {
            "schema_version": manifest.get("schema_version"),
            "conditions": len(manifest.get("conditions", [])),
            "environment": manifest.get("environment"),
        },
        indent=2,
    ))
    display(summary[:8])
    figure = output_dir / "variance_fit.png"
    if figure.is_file():
        from IPython.display import Image
        display(Image(filename=str(figure)))

## 6. Pure-Python verification

This validates encoding, routing shapes, miss-aware pooling, variance-floor recovery, artifacts, and notebook metadata without accessing hardware.

In [ ]:
run_python(VERIFY)

## 7. Mock quick smoke

This runs three potential positions, pool sizes 1 and 4, one placement/routing condition, and eight trials.

In [ ]:
mock_dir = ARTIFACT_ROOT / "mock_quick"
if RUN_MOCK_SMOKE:
    run_python(
        CLI,
        "--backend", "mock",
        "--quick",
        "--output-dir", mock_dir,
    )
    show_artifacts(mock_dir)
else:
    print("Mock smoke disabled")

## 8. Hardware quick smoke

Enable `RUN_HARDWARE_SMOKE` above and rerun the hardware-client setup cell first. Environment/nightly calibration is permitted only for this small compatibility smoke.

In [ ]:
hardware_smoke_dir = ARTIFACT_ROOT / "hardware_smoke"
if RUN_HARDWARE_SMOKE:
    run_python(
        CLI,
        "--backend", "hardware",
        "--quick",
        "--allow-environment-calibration",
        "--output-dir", hardware_smoke_dir,
    )
    show_artifacts(hardware_smoke_dir)
else:
    print("Hardware smoke disabled")

## 9. Operating-point sweep

Enable `RUN_OPERATING_POINT_SWEEP` only after the hardware quick smoke succeeds. This quick sweep evaluates a small threshold/weight/gain grid and writes `selected_operating_point.json`. Expand the lists for the formal calibration sweep.

In [ ]:
operating_point_dir = ARTIFACT_ROOT / "operating_point"
if RUN_OPERATING_POINT_SWEEP:
    run_python(
        CLI,
        "--phase", "calibrate",
        "--backend", "hardware",
        "--quick",
        "--allow-environment-calibration",
        "--calibration-thresholds", 100, 125,
        "--calibration-weights", 47, 63,
        "--calibration-gains", 500,
        "--output-dir", operating_point_dir,
    )
    selected_path = operating_point_dir / "selected_operating_point.json"
    display(json.loads(selected_path.read_text(encoding="utf-8")))
else:
    print("Operating-point sweep disabled")

## 10. Formal pooling experiment

The full experiment requires both an explicit calibration `.pbin` and a selected operating point. It runs pool sizes 1, 2, 4, 8, and 16 across both placement and routing conditions with 256 trials.

In [ ]:
full_dir = ARTIFACT_ROOT / "full_pooling"
selected_path = operating_point_dir / "selected_operating_point.json"

if RUN_FULL_EXPERIMENT:
    if CALIBRATION_PATH is None:
        raise RuntimeError("Set CALIBRATION_PATH to an explicit .pbin before a formal run")
    calibration_path = Path(CALIBRATION_PATH).expanduser().resolve()
    if not calibration_path.is_file():
        raise FileNotFoundError(calibration_path)
    if not selected_path.is_file():
        raise FileNotFoundError(
            f"Run the operating-point sweep first: {selected_path}"
        )
    run_python(
        CLI,
        "--backend", "hardware",
        "--encoding", "identity",
        "--pool-sizes", 1, 2, 4, 8, 16,
        "--placements", "same-quadrant", "cross-quadrant",
        "--routing", "broadcast", "independent",
        "--trials", 256,
        "--calibration", calibration_path,
        "--operating-point-json", selected_path,
        "--output-dir", full_dir,
    )
    show_artifacts(full_dir)
else:
    print("Full experiment disabled")

## 11. Result locations

Download or persist the complete run directory, including `manifest.json`, raw `events.csv`, `events.pt`, summaries, variance fit, and figure.

In [ ]:
for path in sorted(ARTIFACT_ROOT.glob("*")):
    print(path)